# FitCheck 웹 데모 (Kaggle)

인물 사진 + 옷 사진 → 가상 피팅 + 사이즈 추천. 합성 엔진은 FASHN VTON v1.5. Colab 노트북(`run_app.ipynb`)과 같은 앱입니다.

**실행 전 오른쪽 Settings 패널**
- **Accelerator → GPU T4 x2** (앱은 GPU 1장만 씁니다. P100은 1번 셀이 torch를 맞춰 다시 설치해 3분 더 걸립니다)
- **Internet → On** (계정 전화번호 인증이 되어 있어야 켜집니다. 코드·가중치를 받고 공개 링크를 만드는 데 필요)

셀을 위에서부터 차례로 실행하면 마지막에 **공개 링크**(`https://....gradio.live`)가 나옵니다. 링크는 72시간 유효하고,
세션이 끝나거나 셀을 멈추면 끊깁니다. 설치·가중치 받기에 처음 약 5분, 준비 운동(torch.compile)에 1~2분 걸립니다.
GPU 사용 시간은 주 30시간까지 무료입니다 — 시연이 끝나면 오른쪽 위 전원 버튼으로 세션을 끄세요.

In [ ]:
# 1. GPU 확인 — 이 GPU를 지금 torch가 지원하지 않으면(P100 등) 지원하는 버전으로 바꾼다
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
cap = torch.cuda.get_device_capability(0)
if f'sm_{cap[0]}{cap[1]}' not in torch.cuda.get_arch_list():
    !pip install -q torch==2.5.1 torchvision==0.20.1 --index-url https://download.pytorch.org/whl/cu121
    print('torch를 다시 설치했습니다. 위 메뉴 Run → Restart & clear cell outputs 후 2번 셀부터 실행하세요.')
else:
    print('GPU OK:', torch.cuda.get_device_name(0), '/ torch', torch.__version__)

In [ ]:
# 2. 코드 받기 (앱 + FASHN) — /kaggle/working 밖에 둔다(저장 용량·출력 목록에 안 섞이게)
import os

APP = '/kaggle/tmp/vfa'
FASHN = '/kaggle/tmp/fashn-vton-1.5'
WEIGHTS = '/kaggle/tmp/fashn-weights'
os.makedirs('/kaggle/tmp', exist_ok=True)

if not os.path.exists(APP):
    !git clone -q https://github.com/jhjung03eee/virtual-fitting-app.git {APP}
else:
    !git -C {APP} pull -q
if not os.path.exists(FASHN):
    !git clone -q --depth 1 https://github.com/fashn-AI/fashn-vton-1.5.git {FASHN}
!git -C {APP} log --oneline -1
print('완료')

In [ ]:
# 3. 설치 (세션당 1회 약 3분)
#    mediapipe: 사진 자세·거리 검사(app/photo_check.py)와 후처리. 없으면 검사를 건너뛴다
#    scikit-image: 옷 사진 검사(app/garment_check.py). 없으면 검사를 건너뛴다
!pip install -q -e {FASHN}
!pip install -q mediapipe gradio scikit-image easyocr
print('완료')

In [ ]:
# 4. 가중치 내려받기 (약 2GB, 세션당 1회)
if not os.path.exists(os.path.join(WEIGHTS, 'model.safetensors')):
    !python {FASHN}/scripts/download_weights.py --weights-dir {WEIGHTS}
print('완료')

In [ ]:
# 5. 데모 실행 — 출력에 뜨는 "공개 링크 :" 를 새 탭으로 열면 됩니다 (처음 로딩 약 1~2분)
#    멈추려면 이 셀을 중지하세요. 링크도 같이 끊깁니다.
os.environ['FASHN_REPO'] = FASHN
os.environ['FASHN_WEIGHTS'] = WEIGHTS
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
!python {APP}/app/gradio_app.py

## 잘 안 될 때

- **git clone / pip 에서 연결 오류**: Settings → Internet 이 On 인지 확인하세요.
- **공개 링크가 "생성 실패"**: Gradio 공유 서버 문제일 수 있습니다. 5번 셀을 멈췄다가 다시 실행해 보세요.
- **사진을 올렸는데 "다시 찍어주세요"**: 정면 · 팔은 몸에서 주먹 하나 · 머리부터 발끝까지 화면을 꽉 채워 찍은 사진이어야 합니다.
  사진 아래 빨간 표시를 보고 다시 찍어 올리세요.
- **결과가 어색함**: 옷 사진을 배경이 깔끔한 상품 사진으로 바꾸거나, 옷 사진 종류(상품/착용)가 맞는지 확인하세요.
- **메모리 부족**: Run → Restart 후 2번(경로)과 5번 셀만 실행하세요 (설치·가중치는 세션 안에서 남아 있습니다).